[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/two_spins/two_spins_spacetime.ipynb)

# Two Spins in Spacetime

This notebook computes the experiment of [the two spins notebook](two_spins.ipynb), two spins that swap and entangle under their exchange interaction, in two copies of spacetime instead of two copies of space. One spin up and one spin down swap over time, and halfway they are entangled: then neither spin on its own turns in any plane, yet measured in the same plane the two always disagree. One observer, at rest with the spins and the fields, sees both; what that observer calls space is the part of spacetime perpendicular to its time.

Each spin has its own copy of spacetime, with its own time direction, `t` and `T`, both the observer's time: the two spins are separate systems, and only in separate copies does everything that belongs to one spin commute with everything that belongs to the other, since with a shared time direction the planes through it of the two spins would anticommute. Every object of the experiment is spacelike for the observer and lies in the even algebra. A spin shows itself as the plane it turns in, spin up along z as `mv.xy`. The pair's state is a product of the two spins' states in the ideal of the correlator `0.5 * (1 - xy * XY)`, where `correlator * xy` is the reference spin, the spin of the state with both spins up. Measurements are made in planes, a field is a plane, the correlations of the two spins are a map from the second spin's planes to the first's, and the Bell combination is a sum of products of planes, `-(a * (b + b_other) + a_other * (b - b_other))`. No pseudoscalar appears anywhere.

In bra-ket notation the pair's state reads as a ket of two qubits, the plane a spin turns in as the Bloch vector of its reduced density matrix, a unit plane of one spin acting on the pair as the Pauli operator along the normal of that plane, and the observer's frame as the basis in which the Pauli matrices are written.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_animation
from examples.quantum.two_spins import render

np.set_printoptions(precision=4, suppress=True)

# Two copies of spacetime, one per spin: t x y z for the first, T X Y Z for the second. Space squares
# to plus one and time to minus one.
ga = Algebra("t-x+y+z+T-X+Y+Z+")
context = NumpyContext(ga)
mv = context.multivector
one = mv.scalar([1.0])                                                          # [] Scalar
# The planes of each spin's space: spacelike for the observer, whose time is t in the first copy and T
# in the second.
FirstPlanes = ga.gatype.bivector().derive.within("x y z")
SecondPlanes = ga.gatype.bivector().derive.within("X Y Z")
# The state of one spin, and of the pair: a product of the two spins' states.
FirstSpinor = ga.gatype.even().derive.within("x y z")
SecondSpinor = ga.gatype.even().derive.within("X Y Z")
Spinor = (FirstSpinor * SecondSpinor).output_gatype
Correlation = ga.gatype((FirstPlanes, SecondPlanes))
Bivector = ga.gatype.bivector()

## 1. Two spins in two copies of spacetime

The state of one spin is an even multivector of its own space, a scaled rotor: spin up along z is one, and spin down is the half turn that takes the plane xy to its opposite, `-zx`. Such a state commutes with its copy's time direction: it is spacelike for the observer. The state of the pair is a product of the two, taken in the ideal of the correlator `correlator = 0.5 * (1 - xy * XY)`. In that ideal, multiplying on the right by either spin's xy plane is the same: it is `reference_spin = correlator * xy`, which squares to `-correlator`; multiplied from the right, it turns a state about the reference spin. Planes of different spins commute, so each spin's planes act on the pair without disturbing the other's. A state is normalized to one in the scalar part of its density, `2 * (state >> one)`.

In bra-ket notation the pair's state reads as a ket $|\psi\rangle \in \mathbb{C}^2 \otimes \mathbb{C}^2$, spin up and spin down as $|0\rangle$ and $|1\rangle$, and right multiplication by `reference_spin` as multiplication by $i$.

In [ ]:
correlator = 0.5 * (one - mv.xy * mv.XY)                                        # [] Spinor
reference_spin = correlator * mv.xy                                             # [] Spinor
# The first spin up, the second down.
up_down = -mv.ZX * correlator                                                   # [] Spinor
norm = 2 * (up_down >> one).select[0]                                           # [] Scalar

In [ ]:
print("correlator * correlator - correlator:", np.abs((correlator * correlator - correlator).kernel).max())
print("correlator * xy - correlator * XY:", np.abs((correlator * mv.xy - correlator * mv.XY).kernel).max())
print("reference_spin * reference_spin + correlator:", (reference_spin * reference_spin + correlator).scalar_norm_squared().to_array())
print("t * up_down - up_down * t:", np.abs((mv.t * up_down - up_down * mv.t).kernel).max(), "  norm:", norm.to_array())

## 2. Bloch vectors and correlations

The spin of the pair is `2 * (state >> reference_spin)`, a bivector. What one spin shows by itself is the part of it in that spin's own planes: the plane the spin turns in, of size one when the spin is in a state of its own, and smaller when the pair is entangled. What the two spins show together are their correlations in a plane of each: a map from the second spin's planes to the first's, the density `2 * (state >> one)` multiplied by `SecondPlanes`, the space of all the second spin's planes, and kept in the first spin's planes, `(density * SecondPlanes).cast(FirstPlanes)`. Planes pair positively under the dot product, so `a.dot(correlation(b))` is the expected product of the two spins' values in the planes `a` and `b`. The picture shows each spin's plane as a point of the space of its planes, with coordinates along yz, zx and xy: spin up and spin down, each in a state of its own.

The Bell combination of four correlations, in two planes of each spin, stays within 2 in any account in which each spin carries its own answers. For a given state, its largest value over all planes is `2 * sqrt(s1**2 + s2**2)`, from the two largest singular values of the correlation map. Spin up and spin down reach 2 exactly.

In bra-ket notation, with the Pauli matrices $\boldsymbol\sigma$, the plane of one spin reads as the Bloch vector $\operatorname{tr}(\rho_1 \boldsymbol\sigma)$ of its reduced density matrix $\rho_1 = \operatorname{tr}_2 |\psi\rangle\langle\psi|$, the correlation map as the correlation tensor $T_{ij} = \langle\psi| \sigma_i \otimes \sigma_j |\psi\rangle$, and the largest Bell combination as $2\sqrt{t_1^2 + t_2^2}$ over the singular values $t_i$ of $T$.

In [ ]:
def spin_planes(state: Spinor) -> tuple[FirstPlanes, SecondPlanes]:
    """The plane each spin turns in: the part of the pair's spin in that spin's own planes."""
    spin = 2 * (state >> reference_spin)                                        # [...] Bivector
    return spin.cast(FirstPlanes), spin.cast(SecondPlanes)


def correlation(state: Spinor) -> Correlation:
    """The correlations of the two spins, as a map from the second spin's planes to the first's."""
    density = 2 * (state >> one)                                                # [...] Spinor
    return (density * SecondPlanes).cast(FirstPlanes)                           # [...] FirstPlanes <- SecondPlanes


def bell(correlations: Correlation):
    """The largest Bell combination over all planes, from the two largest singular values."""
    values = correlations.svdvals()                                             # [..., 3] Scalar
    return 2 * (values[..., 0] ** 2 + values[..., 1] ** 2).square_root()       # [...] Scalar


start_first, start_second = spin_planes(up_down)                                # [] FirstPlanes, SecondPlanes
start_correlations = correlation(up_down)                                       # [] FirstPlanes <- SecondPlanes
start_bell = bell(start_correlations)                                           # [] Scalar
render.draw_planes(start_first, start_second);

In [ ]:
print("largest Bell combination of spin up and spin down:", start_bell.to_array())

## 3. The exchange

The exchange interaction couples each plane of one spin to the same plane of the other: `coupling = yz * YZ + zx * ZX + xy * XY`. Squared, `coupling * coupling == 3 + 2 * coupling`: the coupling satisfies a quadratic with roots 3 and -1. So `(1 + coupling) / 4` and `(3 - coupling) / 4` are idempotent and add up to one. They split every state into its singlet part, on which the coupling is 3, and its triplet part, on which it is -1. Over time each part turns by its own rotor in the xy plane, multiplied on the right: the singlet part by three times the angle, the triplet part back by the angle. Spin up and spin down are half singlet and half triplet. As the two parts turn apart, the spins swap, and halfway they are fully entangled: neither spin turns in any plane, the correlation map turns every unit plane into its opposite, and the largest Bell combination reaches `2 * sqrt(2)`. The plots follow the size of each spin's plane and the largest Bell combination over the exchange, against the bound of 2 for spins that carry their own answers (dashed) and the bound for any pair (dotted).

The pair's state is a spinor, not one plus a sum of planes and products of planes, because the coupling cannot turn a state of that form by a commutator: a product of planes of the two spins squares to plus one, `(yz * YZ) * (yz * YZ) == 1`, so the commutator with the coupling stretches states instead of turning them. The turn comes from the reference spin, which acts on the spinor from the right.

In bra-ket notation the exchange reads as $e^{-i\theta\,\boldsymbol\sigma_1\cdot\boldsymbol\sigma_2}$, and the two idempotents as the projectors $\tfrac14\left(1 - \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$ onto the singlet and $\tfrac14\left(3 + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$ onto the triplet.

In [ ]:
coupling = mv.yz * mv.YZ + mv.zx * mv.ZX + mv.xy * mv.XY                        # [] Spinor
singlet_part = 0.25 * (one + coupling)                                          # [] Spinor
triplet_part = 0.25 * (3 - coupling)                                            # [] Spinor


def exchange(state: Spinor, angle: np.ndarray) -> Spinor:
    """The state after the exchange has acted for the given angle, the coupling strength times time."""
    return singlet_part * state * (mv.xy * (3 * angle)).exp() + triplet_part * state * (mv.xy * -angle).exp()   # [...] Spinor


n_angles = 121
angles = np.linspace(0.0, np.pi / 4, n_angles)
halfway = n_angles // 2
states = exchange(up_down, angles)                                              # [angles] Spinor
first, second = spin_planes(states)                                             # [angles] FirstPlanes, SecondPlanes
correlations = correlation(states)                                              # [angles] FirstPlanes <- SecondPlanes
bells = bell(correlations)                                                      # [angles] Scalar
render.draw_exchange_planes(angles, bells, first);

In [ ]:
print("coupling * coupling - 3 - 2 * coupling:", np.abs((coupling * coupling - 3 - 2 * coupling).kernel).max())
print("singlet part idempotent:", np.abs((singlet_part * singlet_part - singlet_part).kernel).max(), "  triplet part idempotent:", np.abs((triplet_part * triplet_part - triplet_part).kernel).max())
print("(yz * YZ) * (yz * YZ):", ((mv.yz * mv.YZ) * (mv.yz * mv.YZ)).to_array())
print("largest Bell combination at the start, halfway and the end:", bells.to_array()[[0, halfway, -1]])

## 4. The singlet and the Bell bound

The singlet part of spin up and spin down, normalized, is the singlet: the one state on which the coupling is 3, which the exchange only turns by a phase. Its correlation map turns every plane of the second spin into the opposite plane of the first. For given planes the Bell combination is the expectation of a multivector acting on states from the left, the Bell element: each product of a plane of one spin with a plane of the other, `a * b`, is minus the product of the two spins' values in a and b. Squared, the Bell element is `4 - 4 * a.commutator(a_other) * b.commutator(b_other)`: four, less four times the product of two planes, the commutator of the first spin's two planes and that of the second's. The commutator of two unit planes is a plane of the size of the sine of the angle between them, so neither is larger than one, the square is at most eight, and no state takes the combination past `2 * sqrt(2)`. The singlet reaches it, with the first spin's planes a quarter turn apart and the second spin's halfway between them, reversed.

In bra-ket notation the Bell element reads as the CHSH operator $\mathcal{B} = A \otimes (B + B') + A' \otimes (B - B')$, its square as $\mathcal{B}^2 = 4 - [A, A'] \otimes [B, B']$, and the bound as Tsirelson's.

In [ ]:
def bell_element(first: FirstPlanes, first_other: FirstPlanes, second: SecondPlanes, second_other: SecondPlanes) -> Spinor:
    """The Bell combination in two planes of each spin, as a multivector acting on states from the left."""
    return -(first * (second + second_other) + first_other * (second - second_other))   # [...] Spinor


def expectation(element: Spinor, state: Spinor):
    """The expectation of a multivector acting on states from the left: the sandwich from the other
    side, `state.reverse() * element * state`."""
    return 2 * (state << element).select[0]                                     # [...] Scalar


singlet = np.sqrt(2) * singlet_part * up_down                                   # [] Spinor
singlet_correlations = correlation(singlet)                                     # [] FirstPlanes <- SecondPlanes
# Any four unit planes, and the Bell element's square against the product of the two commutators.
rng = np.random.default_rng(0)
a, a_other = mv(FirstPlanes, rng.normal(size=(2, 3))).normalized()              # [] FirstPlanes each
b, b_other = mv(SecondPlanes, rng.normal(size=(2, 3))).normalized()             # [] SecondPlanes each
square = bell_element(a, a_other, b, b_other) * bell_element(a, a_other, b, b_other)   # [] Spinor
bound = 4 - 4 * a.commutator(a_other) * b.commutator(b_other)                   # [] Spinor
# The planes in which the singlet reaches the bound.
half = 1 / np.sqrt(2)
reached = expectation(bell_element(mv.yz, mv.zx, -half * (mv.YZ + mv.ZX), -half * (mv.YZ - mv.ZX)), singlet)   # [] Scalar

In [ ]:
print("coupling * singlet - 3 * singlet:", np.abs((coupling * singlet - 3 * singlet).kernel).max())
print("correlation of YZ, ZX, XY plus yz, zx, xy:", [np.abs((singlet_correlations(across) + along).kernel).max() for across, along in ((mv.YZ, mv.yz), (mv.ZX, mv.zx), (mv.XY, mv.xy))])
print("square of the Bell element less its bound:", np.abs((square - bound).kernel).max())
print("Bell combination the singlet reaches:", reached.to_array(), "  2 * sqrt(2):", 2 * np.sqrt(2))

## 5. Steering by measurement

Measuring the first spin in a plane leaves the second spin, however far away, in a new state. The first spin's value in a unit plane `n` acts on the pair from the left, `-n * state * reference_spin`, and finding +1 keeps the part of the state on which that value is +1: half the state plus half that product. How likely that is, and the second spin's plane after, follow from what the pair already shows. The probability is `0.5 * (1 + n.dot(first))`, and the second spin's plane is `(second + correlations.adjoint()(n)) / (1 + n.dot(first))`. The correlation map takes the second spin's planes to the first's; its adjoint, the map read backwards, is what steers. Finding -1 in `n` is finding +1 in `-n`, so planes over the whole sphere of unit planes, the turns of `xy`, cover both outcomes.

Each colour in the figure below is one plane on the first spin's sphere of unit planes (left), and where finding +1 in it leaves the second spin's plane (right), sized by how likely it is. At the start the spins are in states of their own: whatever the measurement finds, the second spin stays down. Halfway they are fully entangled, and the measurements leave it anywhere on its sphere. At the end they have swapped, and the second spin stays up.

In bra-ket notation the measurement reads as the projector $\tfrac12(1 + \mathbf n\cdot\boldsymbol\sigma)\otimes 1$, and the steered Bloch vector as $(\mathbf b + T^{\mathsf T}\mathbf n)/(1 + \mathbf n\cdot\mathbf a)$, with $T$ the correlation tensor.

In [ ]:
def steer(state: Spinor, plane: FirstPlanes):
    """Measuring the first spin in a unit plane and finding +1: how likely that is, and the second
    spin's plane after."""
    # The first spin's value in the plane acts on the pair from the left; the measurement keeps the part
    # of the state on which that value is +1.
    measured = 0.5 * (state - plane * state * reference_spin)                   # [...] Spinor
    probability = expectation(one, measured)                                    # [...] Scalar
    # What is left of the pair, normalized again, and the second spin's part of it.
    _, steered = spin_planes(measured / probability.square_root())              # [...] SecondPlanes
    return probability, steered


# Unit planes spread over the first spin's sphere of planes: xy tipped by the centres of equal steps of
# polar angle, then turned in xy by equal steps of azimuth.
n_around, n_down = 24, 12
polar = (np.arange(n_down) + 0.5) / n_down * np.pi
azimuth = np.arange(n_around) / n_around * 2 * np.pi
tipped = (mv.zx * (-polar / 2)).exp() >> mv.xy                                  # [down] FirstPlanes
planes = (mv.xy * (-azimuth / 2)).exp() >> tipped[:, None]                      # [down, around] FirstPlanes
# Each plane measured at every stage of the exchange.
probability, steered = steer(states[:, None, None], planes)                     # [angles, down, around] Scalar, SecondPlanes
# The same, from the correlation map read backwards: its adjoint takes the first spin's planes to the
# second's.
backwards = (second[:, None, None] + correlations[:, None, None].adjoint()(planes)) / (1 + planes.dot(first[:, None, None]))   # [angles, down, around] SecondPlanes
show_animation(render.animate_steering_planes(planes, probability, steered), "two_spins_steering_spacetime", 60)

In [ ]:
print("steered less the correlation map read backwards:", np.abs((steered - backwards).kernel).max())
print("probability less 0.5 * (1 + n.dot(first)):", np.abs((probability - 0.5 * (1 + planes.dot(first[:, None, None]))).kernel).max())

## 6. Entanglement in the singular values

Turning either spin by a rotor of its own space, acting on the pair from the left, turns that spin's plane, and acts on the correlation map from that spin's side: `correlation(first_turn * second_turn * state) == first_turn >> correlations(second_turn << SecondPlanes)`. Turning a map from either side leaves its singular values alone, so nothing done to the spins one at a time can change them: they measure how entangled the pair is. For every state of the pair they are one, and the same value $C$ twice, the concurrence; each spin's plane has size $\sqrt{1 - C^2}$, so the more entangled the pair, the less either spin shows on its own. Over the exchange the concurrence rises from zero to one halfway and falls back, and the largest Bell combination of section 2 is $2\sqrt{1 + C^2}$.

In bra-ket notation the turns read as local unitaries $U_1 \otimes U_2$, acting on the correlation tensor as $T \mapsto R_1 T R_2^{\mathsf T}$, and the concurrence of a pure state as $C = |\langle\psi|\,\sigma_y\otimes\sigma_y\,|\psi^*\rangle|$.

In [ ]:
# Each spin turned by a rotor of its own space, acting on the pair from the left.
first_turn = mv(FirstPlanes, rng.normal(size=3)).exp()                          # [] Rotor
second_turn = mv(SecondPlanes, rng.normal(size=3)).exp()                        # [] Rotor
turned_correlations = correlation(first_turn * second_turn * states)            # [angles] FirstPlanes <- SecondPlanes
# The turns act on the correlation map from either side.
from_either_side = first_turn >> correlations(second_turn << SecondPlanes)      # [angles] FirstPlanes <- SecondPlanes
singular_values = correlations.svdvals()                                         # [angles, 3] Scalar
turned_singular_values = turned_correlations.svdvals()                           # [angles, 3] Scalar
concurrence = singular_values[..., 1]                                            # [angles] Scalar
render.draw_entanglement_planes(angles, singular_values, first);

In [ ]:
print("turned correlation map less the map turned from either side:", np.abs((turned_correlations - from_either_side).kernel).max())
print("singular values at the start, halfway and the end:\n", singular_values.to_array()[[0, halfway, -1]])
print("the same after the turns:\n", turned_singular_values.to_array()[[0, halfway, -1]])
print("plane size squared plus concurrence squared, largest deviation from one:", np.abs(first.scalar_norm_squared().to_array() + concurrence.to_array() ** 2 - 1).max())

## 7. Energy levels in a field

A magnetic field is a plane, the plane it turns spins in. A field in the xy plane of both spins, `field_plane = xy + XY`, turns each spin in that plane, and the energy it adds depends on the spins' planes alone: minus half the field strength times the two spins' parts in its plane. The exchange adds its part through the singlet and triplet idempotents, shifted here to minus the exchange rate on the singlet and plus it on the triplet. In such a field the singlet and the three triplets, the one of zero spin in the field's plane and the ones with both spins up or both down, keep their shapes, and the energy of each is a line in the field. The triplet with both spins up falls with the field until it crosses the singlet at twice the exchange rate. There the lowest state jumps from the singlet, entangled and with no spin in the field's plane, to both spins up.

In bra-ket notation the energy reads as the expectation of $H = \tfrac{J}{2}\left(1 + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right) - \tfrac{B}{2}\left(\sigma_{1z} + \sigma_{2z}\right)$, and the jump as the level crossing of an antiferromagnetic dimer.

In [ ]:
# The plane of a field that turns both spins in their xy planes.
field_plane = mv.xy + mv.XY                                                     # [] Bivector


def energy(state: Spinor, exchange_rate, field):
    """The expected energy of a state: the exchange, shifted to minus the exchange rate on the singlet
    and plus it on the triplet, and a field that lowers the energy of each spin turning in its plane."""
    first, second = spin_planes(state)                                          # [...] FirstPlanes, SecondPlanes
    return exchange_rate * expectation(0.5 * (one - coupling), state) - 0.5 * field * field_plane.dot(first + second)   # [...] Scalar


# The four states of definite spin in the field's plane, one entry per state: the singlet, the triplet of
# zero spin, and both spins up or both down.
four = stack([singlet, np.sqrt(2) * triplet_part * up_down, correlator, mv.zx * mv.ZX * correlator])   # [4] Spinor
four_first, four_second = spin_planes(four)                                     # [4] FirstPlanes, SecondPlanes
spins = field_plane.dot(four_first + four_second)                               # [4] Scalar
exchange_rate = 1.0
n_fields = 201
fields = np.linspace(0.0, 4.0, n_fields)
energies = energy(four, exchange_rate, fields[:, None])                         # [fields, 4] Scalar
render.draw_levels(fields, energies, spins);

In [ ]:
print("spin in the field's plane of the singlet, the zero triplet, both up and both down:", spins.to_array())
print("energies at no field, at twice the exchange rate and at four times it:\n", energies.to_array()[[0, n_fields // 2, -1]])

## 8. A qubit made of two spins

A field in the xy planes that differs between the spins, as for electrons in two neighbouring quantum dots, turns the first spin's xy plane against the second's. It leaves the triplets with both spins up or both down alone, and mixes the singlet with the triplet of zero spin in that plane, while the exchange holds those two apart. On these two states the generator of time, the shifted exchange turned about the reference spin plus the field difference, applied twice is a number: minus the rate squared, the sum of the squares of the exchange rate and the field difference. So its exponential is a cosine and a sine, `cos(rate * time) * state + sin(rate * time) / rate * generator(state)`, as for a rotor, and the pair turns as one qubit between the singlet and the triplet.

Its place on that qubit's sphere has three coordinates: the balance of singlet against triplet, from top to bottom; how far it leans to up-down against down-up, half the difference of the two spins' parts in their xy planes; and the turn between them, the expectation of the shifted exchange times the difference of the two xy planes. From the singlet at the top, the pair circles an axis tilted from vertical by the field difference against the exchange. With the field difference alone (purple) the axis lies flat: the pair turns all the way to the triplet, through states that are all fully entangled, since the field difference only shifts the phase between up-down and down-up. Against an equal exchange rate (red) the circle reaches up-down, a spin up and a spin down each in a state of its own. A stronger exchange (blue) holds the pair near the singlet. Beside the sphere, how likely the pair is found back in the singlet.

In bra-ket notation the generator reads as $-iH$ with $H = \tfrac{J}{2}\left(1 + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right) + \tfrac{\Delta}{2}\left(\sigma_{1z} - \sigma_{2z}\right)$, which on the singlet and the triplet of zero spin squares to $J^2 + \Delta^2$, and the probability of finding the pair back in the singlet as $1 - \tfrac{\Delta^2}{J^2 + \Delta^2}\sin^2\!\left(\sqrt{J^2 + \Delta^2}\,t\right)$. This pair is the singlet–triplet qubit of semiconductor quantum dots, first driven by Petta and coworkers (Science 309, 2180, 2005).

In [ ]:
# The planes of a field difference: the first spin's xy plane against the second's.
difference_plane = 0.5 * (mv.xy - mv.XY)                                        # [] Bivector


def generator(state: Spinor, exchange_rate, field: Bivector) -> Spinor:
    """The generator of time on a state of zero spin in the xy planes: the exchange, minus one on the
    singlet and one on the triplet, turned about the reference spin; and the field, a plane in each spin's
    space, acting from the left."""
    return -exchange_rate * (0.5 * (one - coupling) * state) * reference_spin - field * state # [...] Spinor


def qubit_turn(state: Spinor, exchange_rate, difference, times) -> Spinor:
    """A state of zero spin in the xy planes after the given times: the generator applied twice is minus
    the rate squared, so its exponential is a cosine and a sine."""
    rate = np.sqrt(exchange_rate**2 + difference**2)
    return np.cos(rate * times) * state + np.sin(rate * times) / rate * generator(state, exchange_rate, difference * difference_plane)   # [...] Spinor


def qubit(state: Spinor):
    """The pair's place on the qubit's sphere: how far it leans to up-down against down-up, the turn
    between them, and the balance of singlet against triplet."""
    first, second = spin_planes(state)                                          # [...] FirstPlanes, SecondPlanes
    across = 0.5 * (mv.xy.dot(first) - mv.XY.dot(second))                      # [...] Scalar
    turned = expectation(-0.25 * (one - coupling) * (mv.xy - mv.XY), state)     # [...] Scalar
    balance = expectation(0.5 * (coupling - one), state)                        # [...] Scalar
    return across, turned, balance


# One field difference against three exchange rates, one entry per case, from the singlet.
difference = 1.0
field = difference * difference_plane                                           # [] Bivector
exchange_rates = np.array([0.0, 1.0, 3.0])
times = np.linspace(0.0, np.pi, 241)
qubit_states = qubit_turn(singlet, exchange_rates[:, None], difference, times)  # [rates, times] Spinor
back_in_singlet = expectation(singlet_part, qubit_states)                       # [rates, times] Scalar
across, turned, balance = qubit(qubit_states)                                   # [rates, times] Scalar each
# The generator applied twice, on the singlet and on the triplet of zero spin, for the equal exchange rate.
case = 1
pair = four[:2]                                                                 # [2] Spinor
twice = generator(generator(pair, exchange_rates[case], field), exchange_rates[case], field)   # [2] Spinor
turned_first, turned_second = spin_planes(qubit_states)                         # [rates, times] FirstPlanes, SecondPlanes
render.draw_qubit(across, turned, balance, times, back_in_singlet, exchange_rates);

In [ ]:
rate = np.hypot(exchange_rates[case], difference)
print("generator twice plus the rate squared, on the singlet and the zero triplet:", np.abs((twice + rate**2 * pair).kernel).max())
print("largest distance from the qubit's sphere:", np.abs((across * across + turned * turned + balance * balance).to_array() - 1).max())
print("largest plane over the turn, for each exchange rate:", np.sqrt(turned_first.scalar_norm_squared().to_array()).max(axis=-1))
print("least found back in the singlet, for each exchange rate:", back_in_singlet.to_array().min(axis=-1))

In [ ]:
# checks
# The experiment computed in two copies of space gives the same numbers: Bell combinations, each spin's
# plane against its Bloch vector along the normal, singular values, steering, energies and the qubit.
from examples.quantum.two_spins import core, scenarios

space_states = core.exchange(scenarios.UP_DOWN, angles)                         # [angles] Spinor, in space
space_first, space_second = core.bloch(space_states)                            # [angles] First, Second
space_correlations = core.correlation(space_states)                             # [angles] Correlation
np.testing.assert_allclose(bells.to_array(), core.bell(space_correlations).to_array(), atol=1e-12)
np.testing.assert_allclose(render.components(first, "yz zx xy"), render.components(space_first, "x y z"), atol=1e-12)
np.testing.assert_allclose(render.components(second, "YZ ZX XY"), render.components(space_second, "X Y Z"), atol=1e-12)
np.testing.assert_allclose(singular_values.to_array(), space_correlations.svdvals().to_array(), atol=1e-12)
space_probability, space_steered = core.steer(space_states[:, None, None], scenarios.sphere(n_around, n_down))
np.testing.assert_allclose(probability.to_array(), space_probability.to_array(), atol=1e-12)
np.testing.assert_allclose(render.components(steered, "YZ ZX XY"), render.components(space_steered, "X Y Z"), atol=1e-12)
_, space_reached = scenarios.singlet()                                          # [] Scalar
np.testing.assert_allclose(reached.to_array(), space_reached.to_array(), atol=1e-12)
np.testing.assert_allclose(energies.to_array(), scenarios.field_sweep(fields, exchange_rate).to_array(), atol=1e-12)
space_qubit_states, space_back_in_singlet = scenarios.singlet_return(times, exchange_rates, difference)
np.testing.assert_allclose(back_in_singlet.to_array(), space_back_in_singlet.to_array(), atol=1e-12)
for coordinate, space_coordinate in zip((across, turned, balance), core.qubit(space_qubit_states)):
    np.testing.assert_allclose(coordinate.to_array(), space_coordinate.to_array(), atol=1e-12)

# The correlator is idempotent; the singlet and triplet parts add up to one; the exchange keeps the
# state normalized and swaps the spins through a fully entangled state; the Bell element squares to four
# less four times the product of the two commutators; measuring the first spin steers the second through
# the correlation map read backwards; turning each spin acts on the correlation map from either side; the
# pair returns to the singlet as the closed form says.
np.testing.assert_allclose((correlator * correlator - correlator).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((singlet_part + triplet_part - one).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(expectation(one, states).to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose(first[halfway].kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((first[-1] + mv.xy).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((square - bound).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((steered - backwards).kernel, 0.0, atol=1e-11)
np.testing.assert_allclose((probability - 0.5 * (1 + planes.dot(first[:, None, None]))).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((turned_correlations - from_either_side).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(first.scalar_norm_squared().to_array() + concurrence.to_array() ** 2, 1.0, atol=1e-12)
np.testing.assert_allclose((twice + rate**2 * pair).kernel, 0.0, atol=1e-12)
rates = np.hypot(exchange_rates[:, None], difference)
np.testing.assert_allclose(back_in_singlet.to_array(), 1 - (difference / rates) ** 2 * np.sin(rates * times) ** 2, atol=1e-12)

# Mirrors. A reflection of space is an inversion and a half turn in the plane dual to its normal, and the
# inversion leaves every even spacelike element alone; every state commutes with both time directions.
np.testing.assert_allclose((mv.t * states - states * mv.t).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((mv.T * states - states * mv.T).kernel, 0.0, atol=1e-12)
# On spin planes and fields the mirror is the reflection in its normal, in both spins' spaces: the
# normal x takes xy to -xy, keeps yz and takes zx to -zx.
np.testing.assert_allclose((((mv.x * mv.X) >> (mv.xy + mv.yz + mv.zx)) - (mv.yz - mv.xy - mv.zx)).kernel, 0.0, atol=1e-12)
# On the pair's spinor it is the half turn in the dual plane, in both spins' spaces, from the left: the
# state stays in the correlator's ideal, its spin planes are the mirror images, and the exchange and, in
# the mirrored field, the generator carry over. Normals x, z and one halfway between them.
mirrors = ((mv.x * mv.X, mv.yz * mv.YZ),
           (mv.z * mv.Z, mv.xy * mv.XY),
           ((half * (mv.x + mv.z)) * (half * (mv.X + mv.Z)), (half * (mv.yz + mv.xy)) * (half * (mv.YZ + mv.XY))))
for reflection, half_turn in mirrors:
    mirrored_first, mirrored_second = spin_planes(half_turn * states)           # [angles] FirstPlanes, SecondPlanes
    np.testing.assert_allclose((half_turn * states * correlator - half_turn * states).kernel, 0.0, atol=1e-12)
    np.testing.assert_allclose((mirrored_first - (reflection >> first)).kernel, 0.0, atol=1e-12)
    np.testing.assert_allclose((mirrored_second - (reflection >> second)).kernel, 0.0, atol=1e-12)
    np.testing.assert_allclose((exchange(half_turn * up_down, angles) - half_turn * exchange(up_down, angles)).kernel, 0.0, atol=1e-12)
    np.testing.assert_allclose((generator(half_turn * qubit_states, exchange_rates[:, None], reflection >> field) - half_turn * generator(qubit_states, exchange_rates[:, None], field)).kernel, 0.0, atol=1e-12)